In [2]:
import networkx as nx
import numpy as np
from bngenerator import *
from matplotlib import pyplot as plt
from pgmpy.readwrite.XMLBeliefNetwork import XBNReader, XBNWriter
from pgmpy.readwrite import BIFWriter, BIFReader
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import random
import itertools
import numpy as np
import math
from same_decision_probability_calculation import *
from utils import *
from monte_carlo_sdp import *
import os
import glob
import pyAgrum as gum

In [3]:
def pgmpy_to_pyagrum(pgmpy_bn):
    """Convert a pgmpy BayesianNetwork to a pyAgrum BayesNet, preserving
    state names and CPT values. Returns (gum_bn, name_to_id, id_to_name)."""
    gbn = gum.BayesNet("converted")
    name_to_id = {}

    # 1. Add variables with their state labels
    for node in pgmpy_bn.nodes():
        cpd = pgmpy_bn.get_cpds(node)
        states = cpd.state_names[node]
        var = gum.LabelizedVariable(node, node, 0)
        for s in states:
            var.addLabel(str(s))
        name_to_id[node] = gbn.add(var)

    # 2. Add arcs
    for parent, child in pgmpy_bn.edges():
        gbn.addArc(name_to_id[parent], name_to_id[child])

    # 3. Fill CPTs — pgmpy stores values as (child_card, *parent_cards),
    #    pyAgrum's CPT is indexed [parents..., child]. We need to transpose.
    for node in pgmpy_bn.nodes():
        cpd = pgmpy_bn.get_cpds(node)
        values = cpd.get_values()  # shape: (child_card, prod(parent_cards))
        child_card = len(cpd.state_names[node])

        parents = cpd.variables[1:]  # pgmpy convention: [child, p1, p2, ...]
        if parents:
            parent_cards = [len(cpd.state_names[p]) for p in parents]
            # Reshape to (child_card, p1_card, p2_card, ...)
            reshaped = values.reshape([child_card] + parent_cards)
            # Move child axis to the end → (p1_card, p2_card, ..., child_card)
            transposed = np.moveaxis(reshaped, 0, -1)
            gbn.cpt(name_to_id[node]).fillWith(transposed.flatten().tolist())
        else:
            gbn.cpt(name_to_id[node]).fillWith(values.flatten().tolist())

    id_to_name = {v: k for k, v in name_to_id.items()}
    return gbn, name_to_id, id_to_name

In [7]:
def pgmpy_to_pyagrum(pgmpy_bn):
    """Convert a pgmpy BayesianNetwork to a pyAgrum BayesNet, preserving
    state names and CPT values. Returns (gum_bn, name_to_id, id_to_name)."""
    gbn = gum.BayesNet("converted")
    name_to_id = {}

    # 1. Add variables with their state labels
    for node in pgmpy_bn.nodes():
        cpd = pgmpy_bn.get_cpds(node)
        states = cpd.state_names[node]
        var = gum.LabelizedVariable(node, node, 0)
        for s in states:
            var.addLabel(str(s))
        name_to_id[node] = gbn.add(var)

    # 2. Add arcs based EXACTLY on the CPD's parent order.
    # This guarantees pyAgrum's internal parent sequence matches pgmpy's.
    for node in pgmpy_bn.nodes():
        cpd = pgmpy_bn.get_cpds(node)
        parents = cpd.variables[1:]
        for parent in parents:
            gbn.addArc(name_to_id[parent], name_to_id[node])

    # 3. Fill CPTs
    for node in pgmpy_bn.nodes():
        cpd = pgmpy_bn.get_cpds(node)
        values = cpd.get_values() 
        
        # pgmpy's implicit shape is [child, parent1, parent2, ...]
        shape = [len(cpd.state_names[v]) for v in cpd.variables]
        reshaped = values.reshape(shape)
        
        # pyAgrum expects the first added variable (child) to vary fastest, 
        # then parent1, then parent2. 
        # A simple .transpose() completely reverses the axes to:
        # [..., parent2, parent1, child]
        # Flattening this perfectly matches pyAgrum's expected layout.
        transposed = reshaped.transpose()
        gbn.cpt(name_to_id[node]).fillWith(transposed.flatten().tolist())

    id_to_name = {v: k for k, v in name_to_id.items()}
    return gbn, name_to_id, id_to_name

In [4]:
print("Loading models...")
alarm_model = get_example_model('alarm')
child_model = get_example_model('child')
andes_model = get_example_model('andes')
insurance_model = get_example_model('insurance')
win95pts_model  = get_example_model('win95pts')
alarm_model.name = 'alarm'
child_model.name = 'child'
andes_model.name = 'andes'
insurance_model.name = 'insurance'
win95pts_model.name  = 'win95'

Loading models...


In [10]:
def test_converter():
    print("Loading 'child' model from pgmpy...")
    child_model = get_example_model('child')

    print("Converting to pyAgrum...")
    gbn, name_to_id, id_to_name = pgmpy_to_pyagrum(child_model)

    # Initialize inference engines
    pgmpy_ie = VariableElimination(child_model)
    gum_ie = gum.LazyPropagation(gbn)

    # Define test cases: Target variable and Evidence
    # The Child network contains variables like Disease, LowerBodyO2, BirthAsphyxia, Sick, etc.
    test_cases = [
        {"target": "Disease", "evidence": {}}, # Prior probability
        {"target": "Disease", "evidence": {"LowerBodyO2": "<5"}}, # Single evidence
        {"target": "Disease", "evidence": {"LowerBodyO2": "<5", "BirthAsphyxia": "yes"}}, # Multiple evidence
        {"target": "Grunting", "evidence": {"Disease": "TGA", "Sick": "yes"}} # Different target
    ]

    print("\nStarting inference tests...\n" + "-"*45)

    for i, test in enumerate(test_cases, 1):
        target = test["target"]
        evidence = test["evidence"]

        print(f"Test {i}: P({target} | {evidence})")

        # --- pgmpy Inference ---
        # show_progress=False keeps the console clean during tests
        pgmpy_res = pgmpy_ie.query(variables=[target], evidence=evidence, show_progress=False)
        pgmpy_probs = pgmpy_res.values # Extracts the underlying numpy array

        # --- pyAgrum Inference ---
        gum_ie.setEvidence(evidence)
        gum_ie.makeInference()
        # pyAgrum returns a Potential object; .toarray() converts it to a standard numpy array
        gum_res = gum_ie.posterior(name_to_id[target])
        gum_probs = gum_res.toarray()

        # --- Comparison ---
        # np.allclose handles minor floating point variations between the two engines
        is_match = np.allclose(pgmpy_probs, gum_probs, atol=1e-6)

        if is_match:
            print("✅ Results match!")
            print(f"   pgmpy:   {pgmpy_probs}")
            print(f"   pyAgrum: {gum_probs}")
        else:
            print("❌ Results differ!")
            print(f"   pgmpy:   {pgmpy_probs}")
            print(f"   pyAgrum: {gum_probs}")
        print("-" * 45)

In [11]:
test_converter()

Loading 'child' model from pgmpy...
Converting to pyAgrum...

Starting inference tests...
---------------------------------------------
Test 1: P(Disease | {})
✅ Results match!
   pgmpy:   [0.04755102 0.33306122 0.29132653 0.22622449 0.05091837 0.05091837]
   pyAgrum: [0.04755102 0.33306122 0.29132653 0.22622449 0.05091837 0.05091837]
---------------------------------------------
Test 2: P(Disease | {'LowerBodyO2': '<5'})
✅ Results match!
   pgmpy:   [0.04797166 0.3899628  0.26040498 0.20522424 0.04922902 0.04720729]
   pyAgrum: [0.04797166 0.3899628  0.26040498 0.20522424 0.04922902 0.04720729]
---------------------------------------------
Test 3: P(Disease | {'LowerBodyO2': '<5', 'BirthAsphyxia': 'yes'})
✅ Results match!
   pgmpy:   [0.20031496 0.34872156 0.22185422 0.13509483 0.04799269 0.04602174]
   pyAgrum: [0.20031496 0.34872156 0.22185422 0.13509483 0.04799269 0.04602174]
---------------------------------------------
Test 4: P(Grunting | {'Disease': 'TGA', 'Sick': 'yes'})
✅ Res